# Camada GOLD - Analise base de dados "Viagens a Serviço do Portal da Transparência do Governo Federal"

Este notebook le a camada **SILVER** (ja limpa e tipada) e responde 3 perguntas
de negocio. Para cada uma: a consulta SQL, a tabela com o resultado e um grafico.

**Análises realizadas**

| Pergunta                                              |                                                                
| ----------------------------------------------------- |
| **Os 5 órgãos com maior custo total?**                
| **Os 3 destinos com maior custo médio por viagem?**       
| **A viagem de maior duração e seu custo total?** 



## Conexao com o banco 


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import sys
from pathlib import Path

# Pasta raiz do arquivo
raiz = Path().resolve().parent
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))
import banco

conexao = banco.conectar()

def consultar(sql):
    """Roda um SELECT e devolve o resultado como DataFrame do pandas."""
    return pd.read_sql(sql, conexao)

def reais(valor):
    """Formata um numero como moeda brasileira: 1234.5 -> 'R$ 1.234,50'."""
    texto = f'{valor:,.2f}'
    return 'R$ ' + texto.replace(',', 'X').replace('.', ',').replace('X', '.')

print('Conectado ao MySQL com sucesso.')

## Análise 1 – Os 5 órgãos com maior custo total

Soma total dos gastos com as viagens agrupados pelo órgão superior da viagem e ordenado de forma decrescente (maior ao menor).

Através do gráfico de barras horizontais, é possível visualizar a comparação entre categorias.

In [ ]:
#Query SQL

SQL_A1 = """
SELECT
    nome_orgao_superior,
    ROUND(SUM(valor_total),2) AS custo_total
FROM silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5;
"""
dataframe_1 = consultar(SQL_A1)
df1 = dataframe_1.copy()
df1["custo_total"] = df1["custo_total"].apply(reais)
display(df1)


#Plotagem do gráfico
plt.figure(figsize=(16,8))
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.barh(
    dataframe_1["nome_orgao_superior"],
    dataframe_1["custo_total"],
    color='darkred'
)

# Adiciona o valor ao final de cada barra
for i, valor in enumerate(dataframe_1["custo_total"]):
    plt.text(
        valor,                 # posição X
        i,                     # posição Y
        "  " + reais(valor),   # texto
        va='center',
        fontsize=10
    )

plt.title("Órgãos com maior custo total (TOP 3)")
plt.xlabel("Custo total (R$)")
plt.ylabel("Órgão Superior")

plt.gca().invert_yaxis()

#Remove bordas dos eixos
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


# Desativa a notação científica 
plt.gca().ticklabel_format(style='plain', axis='x')

# Formata o eixo em milhões
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'{x/1_000_000:.0f} mi')
)

plt.tight_layout()
plt.show()



## Análise 2 – Os 3 destinos com maior custo médio por viagem?

A consulta contabiliza o TOP 3 destinos com maior custo por viagem cruzando dados das duas tabelas (viagem e trecho)

O gráfico de pirulito diversifica e como há poucas categorias, esse tipo de visualização facilita a leitura dos valores.

In [ ]:
#Query SQL
SQL_A2 = """
SELECT
    t.destino_cidade,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
    ROUND(AVG(v.valor_total),2) AS custo_medio
FROM silver_viagem v
INNER JOIN silver_trecho t
    ON v.id_viagem = t.id_viagem
GROUP BY t.destino_cidade
ORDER BY custo_medio DESC
LIMIT 3;
"""

df2 = consultar(SQL_A2)
display(df2)

#Plotagem do gráfico
plt.figure(figsize=(10,5))

# Linhas
plt.hlines(
    y=df2["destino_cidade"],
    xmin=0,
    xmax=df2["custo_medio"],
    color="gray",
    linewidth=2
)

# Pontos
plt.scatter(
    df2["custo_medio"],
    df2["destino_cidade"],
    s=180
)

# Valores
for i, valor in enumerate(df2["custo_medio"]):
    plt.text(
        valor,
        i,
        "  " + reais(valor),
        va="center",
        fontsize=10
    )

plt.title("Top 3 destinos com maior custo médio por viagem")
plt.xlabel("Custo médio (R$)")
plt.ylabel("Destino")

plt.gca().invert_yaxis()

# Grade
plt.grid(axis='x', linestyle='--', alpha=0.4)

# Remove bordas
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)

plt.tight_layout()
plt.show()


## Análise 3 – Qual a viagem de maior duração e seu custo total?

Esta consulta filtra a quantidade de dias de cada viagem, ordenando por aquela de maior duração e traz seu respectivo valor.

O gráfico de barras vertical foi utilizado, apesar de não ter comparação por ser somente um registro, permite a visualização perante os eixos de valor e duração.

In [ ]:
#Query SQL
SQL_A3 = """
SELECT
    v.id_viagem,
    v.nome_orgao_superior,
    CONCAT(
        MIN(t.origem_uf),
        ' → ',
        GROUP_CONCAT(t.destino_uf ORDER BY t.sequencia_trecho SEPARATOR ' → ')
    ) AS trajeto,
    v.duracao_dias,
    v.valor_total
FROM silver_viagem v
INNER JOIN silver_trecho t
    ON v.id_viagem = t.id_viagem
GROUP BY
    v.id_viagem,
    v.nome_orgao_superior,
    v.duracao_dias,
    v.valor_total
ORDER BY
    v.duracao_dias DESC,
    v.valor_total DESC
LIMIT 1;
"""

df3 = consultar(SQL_A3)
display(df3)

plt.figure(figsize=(6,6))

plt.bar(
    "Viagem mais longa",
    df3["valor_total"].iloc[0]
)

# Exibe o valor acima da barra
plt.text(
    0,
    df3["valor_total"].iloc[0],
    f"{reais(df3['valor_total'].iloc[0])}\n{df3['duracao_dias'].iloc[0]} dias",
    ha="center",
    va="bottom",
    fontsize=10
)

plt.title("Viagem de maior duração e seu custo total")
plt.ylabel("Custo Total (R$)")

# Remove bordas
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Grade horizontal
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

--
## Camada GOLD Agregada

**Tabela GOLD agregada:** Construção de uma camada GOLD com JOIN + GROUP BY com fucoes de
agregacao (`SUM`, `AVG`, `COUNT`, como tabela e como VIEW)


Criada utilizando JOIN + GROUP BY e fica preparada para responder diversas perguntas de negócio.

| Pergunta                                                          |                                                                             
| -----------------------------------------------------             | 
| **Qual o tipo de pagamento com maior valor médio?**               | 
| **Qual o meio de transporte mais usado nos trechos?**             | 
| **Qual UF de destino aparece em mais trechos?**                   | 







In [ ]:
# TAREFA 1: Criar camadas Gold com JOIN, como TABELA e como VIEW.

# ==============================================================================
#  1. GOLD PAGAMENTOS
# ==============================================================================

SQL_DROP_TABELAS = """
DROP TABLE IF EXISTS gold_pagamentos, gold_trechos, gold_viagens;

"""
SQL_DROP_VIEWS = """ 
DROP VIEW IF EXISTS vw_gold_pagamentos, vw_gold_trechos,vw_gold_viagens;
"""

SQL_G1 = """
CREATE TABLE gold_pagamentos AS

SELECT
    p.tipo_pagamento,
    v.nome_orgao_superior,

    COUNT(DISTINCT p.id_pagamento) AS quantidade_pagamentos,
    COUNT(DISTINCT v.id_viagem) AS quantidade_viagens,

    ROUND(AVG(p.valor),2) AS valor_medio_pagamento,
    ROUND(SUM(p.valor),2) AS valor_total_pagamentos

FROM silver_pagamento p

INNER JOIN silver_viagem v
ON p.id_viagem = v.id_viagem

GROUP BY
    p.tipo_pagamento,
    v.nome_orgao_superior;;
"""
SQL_V1 = """
CREATE VIEW vw_gold_pagamentos AS
SELECT *
FROM gold_pagamentos;;
"""

# ==============================================================================
# 2. GOLD TRECHOS
# ==============================================================================

SQL_G2 = """
CREATE TABLE gold_trechos AS

SELECT

    t.destino_uf,
    t.meio_transporte,
    v.nome_orgao_superior,

    COUNT(*) AS quantidade_trechos,
    COUNT(DISTINCT v.id_viagem) AS quantidade_viagens,

    ROUND(AVG(v.valor_total),2) AS custo_medio_viagem

FROM silver_trecho t

INNER JOIN silver_viagem v
ON t.id_viagem = v.id_viagem

GROUP BY

    t.destino_uf,
    t.meio_transporte,
    v.nome_orgao_superior;
"""
SQL_V2 = """
CREATE VIEW vw_gold_trechos AS
SELECT *
FROM gold_trechos;
"""

# ==============================================================================
# 3. GOLD CUSTOS
# ==============================================================================

SQL_G3 = """
CREATE TABLE gold_viagens AS
SELECT
    v.nome_orgao_superior,

    COUNT(*) AS quantidade_viagens,

    ROUND(AVG(v.valor_total),2) AS custo_medio,

    ROUND(SUM(v.valor_total),2) AS custo_total,

    ROUND(AVG(v.duracao_dias),1) AS duracao_media

FROM silver_viagem v

GROUP BY
    v.nome_orgao_superior;
"""

SQL_V3 = """
CREATE VIEW vw_gold_viagens AS
SELECT *
FROM gold_viagens;
"""


def camada_gold():

    print("=== CRIAÇÃO DAS TABELAS CAMADA GOLD ===")

    try:

        conexao = banco.conectar()
        print(" ==== [1/3] Apaga tabelas caso já existam ====")
        banco.executar(conexao, SQL_DROP_TABELAS)
      
        print(" ==== [2/3] Apaga views caso já existam ====")
        banco.executar(conexao, SQL_DROP_VIEWS)
    
        print(" ==== [3/3] Cria as tabelas e as views da GOLD AGREGADA ====")
               
        banco.executar(conexao, SQL_G1)
        print("  Tabela gold_pagamentos OK")
        banco.executar(conexao, SQL_V1)        
        print("  View vw_gold_pagamentos OK")

        banco.executar(conexao, SQL_G2)
        print("  Tabela gold_trechos OK")
        banco.executar(conexao, SQL_V2)        
        print("  View vw_gold_trechos OK")

        banco.executar(conexao, SQL_G3)
        print("  Tabela gold_viagens OK")
        banco.executar(conexao, SQL_V3)       
        print("  View vw_gold_viagens OK")        

        conexao.close()

        print("=== CRIAÇÃO GOLD AGREGADA CONCLUÍDA ===")

    except Exception as erro:

        print("[ERRO]", erro)
        raise


camada_gold()

# Análise 1 - Gold "Qual o tipo de pagamento com maior valor médio?"

In [ ]:
SQL_A4 = """
SELECT
    tipo_pagamento,
    AVG(valor_medio_pagamento) AS valor_medio
FROM gold_pagamentos
GROUP BY tipo_pagamento
ORDER BY valor_medio DESC
LIMIT 3;
"""

df4 = consultar(SQL_A4)
display(df4)

plt.figure(figsize=(9,5))

plt.barh(
    df4["tipo_pagamento"],
    df4["valor_medio"]
)

for i, valor in enumerate(df4["valor_medio"]):
    plt.text(
        valor,
        i,
        "  " + reais(valor),
        va='center',
        fontsize=10
    )

plt.title("Valor médio por tipo de pagamento")
plt.xlabel("Valor médio (R$)")
plt.ylabel("Tipo de pagamento")

plt.gca().invert_yaxis()

plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.show()




# Análise 2 - Gold "Qual o meio de transporte mais usado nos trechos?


In [ ]:
SQL_A5 = """
SELECT

    meio_transporte,
    SUM(quantidade_trechos) AS total

FROM gold_trechos
GROUP BY meio_transporte
ORDER BY total DESC
LIMIT 3;
"""

df5 = consultar(SQL_A5)
display(df5)

plt.figure(figsize=(7,7))

plt.pie(
    df5["total"],
    labels=df5["meio_transporte"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"width":0.45},
    textprops={"fontsize":10}
)

plt.title("Distribuição dos meios de transporte utilizados")

plt.tight_layout()
plt.show()

# Análise 3 - Gold "Qual UF de destino aparece em mais trechos?"

In [ ]:
SQL_A6 = """
SELECT

    destino_uf,
    SUM(quantidade_trechos) AS total

FROM gold_trechos
GROUP BY destino_uf
ORDER BY total DESC
LIMIT 3;

"""

df6 = consultar(SQL_A6)
display(df6)


plt.figure(figsize=(10,6))

plt.hlines(
    y=df6["destino_uf"],
    xmin=0,
    xmax=df6["total"],
    linewidth=2
)

plt.scatter(
    df6["total"],
    df6["destino_uf"],
    s=180
)

for i, valor in enumerate(df6["total"]):
    plt.text(
        valor,
        i,
        f"  {int(valor)}",
        va="center",
        fontsize=10
    )

plt.title("Quantidade de trechos por UF de destino")
plt.xlabel("Quantidade de trechos")
plt.ylabel("UF")

plt.gca().invert_yaxis()

plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# Encerra a conexao com o banco
conexao.close()
print('Conexao encerrada.')